# Rosen-Roback Model API

> High-level API, dataset abstractions, and counterfactual analysis


In [ ]:
# | default_exp model

In [8]:
# | export
from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import Any

import numpy as np
import pandas as pd

from src.equilibrium import EquilibriumSolver, calculate_gdp_change
from src.parameters import CityParameters, EquilibriumResult, ModelParameters

## Overview

This module provides:

1. **Dataset abstractions** - Flexible interface for different data sources
2. **RosenRobackModel** - High-level API for running equilibrium analysis
3. **Counterfactual tools** - Compare baseline vs. policy scenarios

The design follows the **strategy pattern**: any dataset implementing `RosenRobackDataset` can be used with the model, allowing seamless switching between UK data, US data, or synthetic examples.

## Dataset Interface

All datasets must implement `get_cities()` and `get_total_population()`:


In [ ]:
# | export
class RosenRobackDataset(ABC):
    """Abstract base class for datasets usable with the Rosen-Roback model."""

    @property
    @abstractmethod
    def name(self) -> str:
        """Human-readable name for this dataset."""

    @abstractmethod
    def get_cities(self) -> list[CityParameters]:
        """Extract city parameters from the dataset."""

    @abstractmethod
    def get_total_population(self) -> float:
        """Get the total population to distribute across cities."""

    def get_city_names(self) -> list[str]:
        """Get list of city names."""
        return [c.name for c in self.get_cities()]

    def __len__(self) -> int:
        """Number of cities in the dataset."""
        return len(self.get_cities())

    def __repr__(self) -> str:
        return f"{self.__class__.__name__}(name='{self.name}', n_cities={len(self)})"

## Concrete Dataset Implementations

### 1. Synthetic Dataset

Generates random cities for testing and teaching:


In [ ]:
# | export
class SyntheticDataset(RosenRobackDataset):
    """Generate synthetic cities for testing and teaching."""

    def __init__(
        self,
        n_cities: int = 10,  # Number of cities to generate
        total_population: float = 10_000_000,  # Total population across all cities
        tfp_range: tuple[float, float] = (70, 130),  # Range for base TFP values
        amenity_range: tuple[float, float] = (0.7, 1.3),  # Range for amenity values
        elasticity_range: tuple[float, float] = (0.5, 3.0),  # Range for supply elasticity
        supply_shifter: float = 1000.0,  # Supply shifter for all cities
        seed: int | None = None,  # Random seed for reproducibility
    ):
        self._n_cities = n_cities
        self._total_population = total_population
        self._supply_shifter = supply_shifter
        self._cities = self._generate_cities(
            n_cities, tfp_range, amenity_range, elasticity_range, supply_shifter, seed
        )

    def _generate_cities(
        self,
        n: int,
        tfp_range: tuple[float, float],
        amenity_range: tuple[float, float],
        elasticity_range: tuple[float, float],
        supply_shifter: float,
        seed: int | None,
    ) -> list[CityParameters]:
        """Generate random city parameters."""
        rng = np.random.default_rng(seed)
        cities = []
        for i in range(n):
            cities.append(
                CityParameters(
                    name=f"City_{i + 1}",
                    base_tfp=rng.uniform(*tfp_range),
                    amenity=rng.uniform(*amenity_range),
                    supply_elasticity=rng.uniform(*elasticity_range),
                    supply_shifter=supply_shifter,
                )
            )
        return cities

    @property
    def name(self) -> str:
        return f"Synthetic ({self._n_cities} cities)"

    def get_cities(self) -> list[CityParameters]:
        return self._cities

    def get_total_population(self) -> float:
        return self._total_population

### 2. Three-City Dataset

Simple pedagogical example with three cities:


In [ ]:
# | export
class ThreeCityDataset(RosenRobackDataset):
    """Simple three-city dataset for pedagogical examples."""

    def __init__(
        self,
        high_tfp: float = 120,  # TFP for high-productivity city
        medium_tfp: float = 100,  # TFP for medium-productivity city
        low_tfp: float = 80,  # TFP for low-productivity city
        high_elasticity: float = 1.5,  # Supply elasticity for high-productivity city
        medium_elasticity: float = 2.0,  # Supply elasticity for medium-productivity city
        low_elasticity: float = 3.0,  # Supply elasticity for low-productivity city
        total_population: float = 3_000_000,  # Total population to distribute
    ):
        self._total_population = total_population
        self._cities = [
            CityParameters(
                name="High Productivity",
                base_tfp=high_tfp,
                amenity=1.0,
                supply_elasticity=high_elasticity,
                supply_shifter=1000,
            ),
            CityParameters(
                name="Medium",
                base_tfp=medium_tfp,
                amenity=1.0,
                supply_elasticity=medium_elasticity,
                supply_shifter=1000,
            ),
            CityParameters(
                name="Low Productivity",
                base_tfp=low_tfp,
                amenity=1.0,
                supply_elasticity=low_elasticity,
                supply_shifter=1000,
            ),
        ]

    @property
    def name(self) -> str:
        return "Three Cities (HP/M/LP)"

    def get_cities(self) -> list[CityParameters]:
        return self._cities

    def get_total_population(self) -> float:
        return self._total_population

### 3. DataFrame Dataset

Most flexible - works with any pandas DataFrame:


In [ ]:
# | export
class DataFrameDataset(RosenRobackDataset):
    """Dataset backed by a pandas DataFrame with configurable column names."""

    def __init__(
        self,
        df: pd.DataFrame,  # DataFrame containing city data
        name: str,  # Human-readable name for the dataset
        name_col: str = "city",  # Column name for city names
        tfp_col: str | None = None,  # Column name for TFP values
        wage_col: str | None = None,  # Column name for wage values
        rent_col: str | None = None,  # Column name for rent values
        elasticity_col: str = "elasticity",  # Column name for supply elasticity
        population_col: str = "population",  # Column name for population
        default_elasticity: float = 2.0,  # Default elasticity if column not present
    ):
        self._df = df.copy()
        self._name = name
        self._name_col = name_col
        self._tfp_col = tfp_col
        self._wage_col = wage_col
        self._rent_col = rent_col
        self._elasticity_col = elasticity_col
        self._population_col = population_col
        self._default_elasticity = default_elasticity
        self._validate()

    def _validate(self) -> None:
        """Validate that required columns exist."""
        required = [self._name_col, self._population_col]
        missing = [col for col in required if col not in self._df.columns]
        if missing:
            raise ValueError(f"Missing required columns: {missing}")
        if self._tfp_col is None and (self._wage_col is None or self._rent_col is None):
            raise ValueError("Must provide either tfp_col or both wage_col and rent_col")

    @property
    def name(self) -> str:
        return self._name

    def get_cities(self) -> list[CityParameters]:
        """Extract city parameters from the DataFrame."""
        cities = []
        for _, row in self._df.iterrows():
            # Get TFP - either directly or infer from wages/rents
            if self._tfp_col and self._tfp_col in self._df.columns:
                base_tfp = float(row[self._tfp_col])
            else:
                wage = float(row[self._wage_col])
                rent = float(row[self._rent_col])
                base_tfp = wage / (rent**0.33)  # Using beta=0.33

            # Get elasticity
            elasticity = (
                float(row[self._elasticity_col])
                if self._elasticity_col in self._df.columns
                else self._default_elasticity
            )

            cities.append(
                CityParameters(
                    name=str(row[self._name_col]),
                    base_tfp=base_tfp,
                    amenity=1.0,
                    supply_elasticity=elasticity,
                    supply_shifter=1000.0,
                )
            )
        return cities

    def get_total_population(self) -> float:
        return float(self._df[self._population_col].sum())

## Counterfactual Scenarios


In [ ]:
# | export
@dataclass
class CounterfactualScenario:
    """Defines a counterfactual scenario for policy analysis."""

    name: str  # Short name for the scenario
    description: str  # Longer description of what changes
    elasticity_changes: dict[str, float]  # Map from city name to new elasticity

    def apply_to_dataset(
        self,
        baseline: RosenRobackDataset,  # Baseline dataset to modify
    ) -> list[CityParameters]:
        """Apply this scenario to a baseline dataset, returning modified city parameters."""
        cities = baseline.get_cities()
        new_cities = []
        for city in cities:
            if city.name in self.elasticity_changes:
                new_elasticity = self.elasticity_changes[city.name]
                new_cities.append(
                    CityParameters(
                        name=city.name,
                        base_tfp=city.base_tfp,
                        amenity=city.amenity,
                        supply_elasticity=new_elasticity,
                        supply_shifter=city.supply_shifter,
                    )
                )
            else:
                new_cities.append(city)
        return new_cities

## The Rosen-Roback Model

This is the main class users interact with:


In [ ]:
# | export
class RosenRobackModel:
    """Multi-city Rosen-Roback spatial equilibrium model."""

    def __init__(
        self,
        params: ModelParameters | None = None,  # Model parameters (uses STANDARD if None)
    ):
        self.params = params or ModelParameters.STANDARD

    def solve(
        self,
        dataset: RosenRobackDataset,  # Dataset providing city parameters and total population
        **solver_kwargs: Any,  # Additional arguments passed to solver (tol, max_iter, damping, verbose)
    ) -> EquilibriumResult:
        """Solve for spatial equilibrium."""
        cities = dataset.get_cities()
        total_population = dataset.get_total_population()
        solver = EquilibriumSolver(self.params, cities)
        return solver.solve(total_population, **solver_kwargs)

    def solve_with_cities(
        self,
        cities: list[CityParameters],  # Explicit city parameters
        total_population: float,  # Total population to distribute
        **solver_kwargs: Any,  # Additional arguments passed to solver
    ) -> EquilibriumResult:
        """Solve equilibrium given explicit city parameters."""
        solver = EquilibriumSolver(self.params, cities)
        return solver.solve(total_population, **solver_kwargs)

    def run_counterfactual(
        self,
        dataset: RosenRobackDataset,  # Baseline dataset
        scenario: CounterfactualScenario,  # Counterfactual scenario to apply
        **solver_kwargs: Any,  # Additional arguments passed to solver
    ) -> tuple[EquilibriumResult, EquilibriumResult, dict[str, float]]:
        """Run a counterfactual analysis comparing baseline to scenario."""
        baseline = self.solve(dataset, **solver_kwargs)
        counterfactual_cities = scenario.apply_to_dataset(dataset)
        counterfactual = self.solve_with_cities(counterfactual_cities, dataset.get_total_population(), **solver_kwargs)
        gdp_comparison = calculate_gdp_change(baseline, counterfactual)
        return baseline, counterfactual, gdp_comparison

## Tests


In [ ]:
# | hide
# Test SyntheticDataset
dataset = SyntheticDataset(n_cities=5, seed=42)
assert len(dataset) == 5
assert dataset.get_total_population() == 10_000_000
cities = dataset.get_cities()
assert len(cities) == 5
assert all(70 <= c.base_tfp <= 130 for c in cities)

# Test ThreeCityDataset
dataset3 = ThreeCityDataset()
assert len(dataset3) == 3
assert dataset3.get_total_population() == 3_000_000
cities3 = dataset3.get_cities()
assert cities3[0].name == "High Productivity"
assert cities3[0].base_tfp == 120

# Test RosenRobackModel
model = RosenRobackModel()
result = model.solve(dataset3)
assert result.converged
assert len(result.population) == 3
assert np.isclose(sum(result.population), 3_000_000)

# Test counterfactual
scenario = CounterfactualScenario(
    name="Reform",
    description="Increase elasticity in HP",
    elasticity_changes={"High Productivity": 3.0},
)
baseline, counterfactual, gdp = model.run_counterfactual(dataset3, scenario)
assert gdp["gdp_change"] > 0  # Should increase GDP

## Example Usage


In [ ]:
# Create a model with standard parameters
model = RosenRobackModel(params=ModelParameters.STANDARD)

# Use three-city dataset
dataset = ThreeCityDataset()

# Solve baseline
result = model.solve(dataset)
print(result.summary())

# Run counterfactual: increase HP elasticity from 1.5 to 3.0
scenario = CounterfactualScenario(
    name="Housing Reform",
    description="Liberalize high-productivity city",
    elasticity_changes={"High Productivity": 3.0},
)

baseline, counterfactual, gdp_impact = model.run_counterfactual(dataset, scenario)
print(f"\nGDP Impact: {gdp_impact['gdp_change_pct']:.2f}%")

## Key Insights

1. **Dataset abstraction** enables flexibility - same model works with UK, US, or synthetic data
2. **RosenRobackModel** provides clean API - users don't need to know about solvers or log-space
3. **Counterfactual analysis** is straightforward - just specify which cities to reform
4. **Type safety** - All parameters are typed, catching errors at development time

This design makes it easy to:

- Test the model with synthetic data
- Run actual replications with real data
- Compare policy scenarios
- Extend with new datasets (just implement the interface)


In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()